# 02 — Normalización y Clasificación

**Fase 2 del proyecto ISO 20022 en Redes DLT.**

Toma los JSONs crudos de `data/raw/`, decodifica los campos memo (hex en XRPL, texto en Stellar, Base64 en Hedera), unifica los esquemas de las tres redes y etiqueta cada transacción con su código de propósito **ISO 20022** mediante expresiones regulares y NLP. El dataset final se guarda en `data/processed/txs_etiquetadas.csv`.


# 1. Configuración

**sentence-transformers** se importa de forma diferida dentro de **construir_modelo_nlp() **para no exigirlo si **USAR_NLP=False**

In [ ]:
DEBUG: bool = True       # activa prints de verificación
SAVE_DATA: bool = False  # activa guardado de archivos
USAR_NLP: bool = True     # activa clasificación semántica

# 2. Imports

In [ ]:
import os
import re
import glob
import json
import base64
import binascii
import codecs
import pandas as pd
from datetime import datetime, timezone
from typing import Any
from google.colab import drive

drive.mount('/content/drive')

# 3. Paths y Constantes

In [ ]:
# Paths de entrada / salida
BASE_DIR: str = "/content/drive/MyDrive/Tesis/main"
DIR_RAW: dict[str, str] = {
    "stellar": os.path.join(BASE_DIR, "data", "raw", "stellar"),
    "hedera": os.path.join(BASE_DIR, "data", "raw", "hedera"),
    "algorand": os.path.join(BASE_DIR, "data", "raw", "algorand"),}

DIR_PROCESSED: str = os.path.join(BASE_DIR, "data", "processed")
ARCHIVO_SALIDA: str = os.path.join(DIR_PROCESSED, "txs_etiquetadas.csv")

# Esquema del dataset
COLUMNAS: list[str] = [
    "tx_id", "red", "timestamp", "emisor", "receptor",
    "amount", "currency", "fee", "memo_raw",
    "memo_decoded", "iso_code", "iso_metodo",]


# Clasificación NLP
UMBRAL_NLP: float = 0.45

# Patrones regex por código ISO 20022
ISO_REGEX: dict[str, str] = {
    "LQDT": r"lqdt|liquid|liquidez",
    "COLL": r"coll|collateral|colateral|margin",
    "TREA": r"trea|treasur|tesorer",
    "INTC": r"intc|intra.?comp|intercompany",
    "CASH": r"cash|efectivo",}

# Orden de evaluación (de más específico a más amplio)
ORDEN_ISO: list[str] = ["LQDT", "COLL", "TREA", "INTC", "CASH"]

# Descripciones de referencia
ISO_DESCRIPCIONES: dict[str, str] = {
    "LQDT": "liquidity management funding liquidity",
    "COLL": "collateral management margin pledge",
    "TREA": "treasury trading treasury operations",
    "INTC": "intra company internal transfer payment",
    "CASH": "cash management cash pooling",}

# 4. Funciones

In [ ]:
def decodificar_hex(memo: str) -> str:
    """
    Decodifica un campo memo en formato hexadecimal a UTF-8.

    Args:
        memo: string hexadecimal crudo.

    Returns:
        Texto decodificado o string vacío si falla.
    """
    if not memo:
        return ""
    try:
        crudo = binascii.unhexlify(memo)
    except (binascii.Error, ValueError):
        return ""
    return codecs.decode(crudo, "utf-8", "replace")


def decodificar_base64(memo: str) -> str:
    """
    Decodifica un campo memo en formato Base64 a UTF-8.

    Args:
        memo: string Base64 crudo.

    Returns:
        Texto decodificado o string vacío si falla.
    """
    if not memo:
        return ""
    try:
        crudo = base64.b64decode(memo)
    except (binascii.Error, ValueError):
        return ""
    return codecs.decode(crudo, "utf-8", "replace")

In [ ]:
def stroops_a_xlm(valor: Any) -> float:
    """Convierte stroops a XLM (1 XLM = 1e7 stroops)."""
    try:
        return int(valor) / 10_000_000
    except (ValueError, TypeError):
        return float("nan")


def _stellar_memo(tx: dict) -> tuple[str, str]:
    """Devuelve (memo_raw, memo_decoded) de una tx Stellar."""
    tipo = tx.get("memo_type", "none")
    raw = tx.get("memo", "")
    if not raw or tipo == "none":
        return "", ""
    if tipo in ("hash", "return"):
        return raw, decodificar_base64(raw)
    return raw, raw


def normalizar_stellar(tx: dict) -> dict:
    """Mapea una transacción Stellar cruda al esquema."""
    memo_raw, memo_dec = _stellar_memo(tx)
    return {
        "tx_id": tx.get("hash", tx.get("id", "")),
        "red": "stellar",
        "timestamp": tx.get("created_at", ""),
        "emisor": tx.get("source_account", ""),
        "receptor": "",
        "amount": float("nan"),
        "currency": "",
        "fee": stroops_a_xlm(tx.get("fee_charged")),
        "memo_raw": memo_raw,
        "memo_decoded": memo_dec,}

In [ ]:
def hedera_ts_iso(cts: Any) -> str:
    """Convierte un consensus_timestamp Hedera a ISO UTC."""
    if not cts:
        return ""
    try:
        segundos = float(cts)
    except (ValueError, TypeError):
        return ""
    return datetime.fromtimestamp(
        segundos, tz=timezone.utc).isoformat()


def tinybar_a_hbar(valor: Any) -> float:
    """Convierte tinybars a HBAR (1 HBAR = 1e8 tinybars)."""
    try:
        return int(valor) / 100_000_000
    except (ValueError, TypeError):
        return float("nan")


def _hedera_flujo(transfers: list) -> tuple[str, str, float]:
    """Deriva (emisor, receptor, monto) desde transfers."""
    if not transfers:
        return "", "", float("nan")
    emisor = min(transfers, key=lambda t: t.get("amount", 0))
    receptor = max(transfers, key=lambda t: t.get("amount", 0))
    monto = receptor.get("amount", 0) / 100_000_000
    return (emisor.get("account", ""),
            receptor.get("account", ""), monto)


def normalizar_hedera(tx: dict) -> dict:
    """Mapea una transacción Hedera cruda al esquema."""
    emisor, receptor, amount = _hedera_flujo(
        tx.get("transfers", []))
    memo_raw = tx.get("memo_base64", "")
    return {
        "tx_id": tx.get("transaction_id", ""),
        "red": "hedera",
        "timestamp": hedera_ts_iso(
            tx.get("consensus_timestamp")),
        "emisor": emisor,
        "receptor": receptor,
        "amount": amount,
        "currency": "HBAR",
        "fee": tinybar_a_hbar(tx.get("charged_tx_fee")),
        "memo_raw": memo_raw,
        "memo_decoded": decodificar_base64(memo_raw),}

In [ ]:
def microalgos_a_algo(valor: Any) -> float:
    """Convierte microAlgos a ALGO (1 ALGO = 1e6 microAlgos)."""
    try:
        return int(valor) / 1_000_000
    except (ValueError, TypeError):
        return float("nan")


def normalizar_algorand(tx: dict) -> dict:
    """Mapea una transacción Algorand cruda al esquema."""
    pago = tx.get("payment-transaction", {}) or {}
    ts = tx.get("round-time")
    memo_raw = tx.get("note", "")
    return {
        "tx_id": tx.get("id", ""),
        "red": "algorand",
        "timestamp": hedera_ts_iso(ts),  
        "emisor": tx.get("sender", ""),
        "receptor": pago.get("receiver", ""),
        "amount": microalgos_a_algo(pago.get("amount")),
        "currency": "ALGO",
        "fee": microalgos_a_algo(tx.get("fee")),
        "memo_raw": memo_raw,
        "memo_decoded": decodificar_base64(memo_raw),}

In [ ]:
def cargar_ultimo(directorio: str) -> list[dict]:
    """
    Carga el JSON más reciente de un directorio de crudos.
    Args:
        directorio: carpeta con archivos .json por red.
    Returns:
        Lista de transacciones crudas, o [] si no hay.
    """
    archivos = sorted(glob.glob(
        os.path.join(directorio, "*.json")))
    if not archivos:
        if DEBUG:
            print(f"[DEBUG] sin crudos en {directorio}")
        return []
    with open(archivos[-1], encoding="utf-8") as f:
        return json.load(f)

In [ ]:
def clasificar_regex(texto: str) -> str | None:
    """
    Asigna un código ISO por coincidencia de patrones.
    Args:
        texto: memo decodificado.
    Returns:
        Código ISO 20022 o None si no hay coincidencia.
    """
    if not texto:
        return None
    minus = texto.lower()
    for codigo in ORDEN_ISO:
        if re.search(ISO_REGEX[codigo], minus):
            return codigo
    return None


def construir_modelo_nlp() -> dict | None:
    """
    Prepara el modelo semántico y los embeddings de las
    descripciones ISO. Devuelve None si NLP está desactivado
    o la librería no está disponible.
    """
    if not USAR_NLP:
        return None
    try:
        from sentence_transformers import (
            SentenceTransformer, util)
    except ImportError:
        if DEBUG:
            print("[DEBUG] sentence-transformers ausente")
        return None
    modelo = SentenceTransformer("all-MiniLM-L6-v2")
    codigos = list(ISO_DESCRIPCIONES.keys())
    emb = modelo.encode(
        list(ISO_DESCRIPCIONES.values()),
        convert_to_tensor=True)
    return {
        "modelo": modelo,
        "util": util,
        "codigos": codigos,
        "emb": emb,}


def clasificar_nlp(texto: str, ctx: dict) -> str | None:
    """
    Asigna un código ISO por similitud semántica.
    Args:
        texto: memo decodificado.
        ctx: contexto devuelto por construir_modelo_nlp().
    Returns:
        Código ISO si supera UMBRAL_NLP, si no None.
    """
    if not ctx or not texto:
        return None
    emb_txt = ctx["modelo"].encode(
        texto, convert_to_tensor=True)
    sims = ctx["util"].cos_sim(emb_txt, ctx["emb"])[0]
    idx = int(sims.argmax())
    if float(sims[idx]) >= UMBRAL_NLP:
        return ctx["codigos"][idx]
    return None


def clasificar(texto: str, ctx_nlp: dict | None) -> tuple:
    """
    Clasifica un memo: primero regex, luego NLP.
    Returns:
        Tupla (iso_code, iso_metodo). Sin match -> (None, 'null').
    """
    if not texto:
        return None, "null"
    codigo = clasificar_regex(texto)
    if codigo:
        return codigo, "regex"
    if ctx_nlp:
        codigo = clasificar_nlp(texto, ctx_nlp)
        if codigo:
            return codigo, "nlp"
    return None, "null"

# 5. Proceso

In [ ]:
# Cargar y normalizar cada red al esquema unificado
filas: list[dict] = []
for tx in cargar_ultimo(DIR_RAW["stellar"]):
    filas.append(normalizar_stellar(tx))
for tx in cargar_ultimo(DIR_RAW["hedera"]):
    filas.append(normalizar_hedera(tx))
for tx in cargar_ultimo(DIR_RAW["algorand"]):
    filas.append(normalizar_algorand(tx))

df = pd.DataFrame(filas)

if not df.empty:
    ctx_nlp = construir_modelo_nlp()
    etiquetas = df["memo_decoded"].apply(lambda t: clasificar(t, ctx_nlp))
    df["iso_code"] = [e[0] for e in etiquetas]
    df["iso_metodo"] = [e[1] for e in etiquetas]

    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True, errors="coerce")
    df["amount"] = pd.to_numeric(df["amount"], errors="coerce")
    df["fee"] = pd.to_numeric(df["fee"], errors="coerce")

# Orden final de columnas
df = df.reindex(columns=COLUMNAS)

if DEBUG:
    print(f"[DEBUG] filas totales: {len(df)}")
    if not df.empty:
        print("[DEBUG] por red:")
        print(df["red"].value_counts())
        print("[DEBUG] por método de clasificación:")
        print(df["iso_metodo"].value_counts())
        print(df.head())

# 6. Save

In [ ]:
# Guardado del dataset etiquetado
if SAVE_DATA:
    os.makedirs(DIR_PROCESSED, exist_ok=True)
    df.to_csv(ARCHIVO_SALIDA, index=False)
    if DEBUG:
        print(f"[DEBUG] guardado: {ARCHIVO_SALIDA}")

# Backlog

> **Limitaciones v1 (documentadas):** a nivel *transacción*, Stellar/Horizon no expone `amount` ni destino (viven en `/operations`); esos campos quedan vacíos y se enriquecerán en una iteración posterior. En XRPL el `timestamp` depende del `close_time` del ledger: si viene vacío, conviene adjuntarlo en la Fase 1.